In [100]:
import anthropic
import os
from dotenv import load_dotenv
import json
from datetime import datetime
from pprint import pprint
from prompts.emailer import RETENTION_EMAIL_SYSTEM_PROMPT 

load_dotenv()

key = os.environ.get("ANTHROPIC_API_KEY")
assert key is not None, "ANTHROPIC_API_KEY environment variable is not set"
assert RETENTION_EMAIL_SYSTEM_PROMPT is not None, "RETENTION_EMAIL_SYSTEM_PROMPT is not set"

In [ ]:
# See Anthropic API documentation for more details: https://docs.anthropic.com/claude/reference

In [76]:
client = anthropic.Anthropic()

In [77]:
from datetime import datetime
from zoneinfo import ZoneInfo

def get_current_datetime(timezone: str = "UTC") -> str:
    """Return the current date and time in the specified timezone."""
    try:
        now = datetime.now(ZoneInfo(timezone))
        return now.strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception as e:
        return f"Error: {str(e)}"
    
def format_datetime(dt: datetime) -> str:
    """Format a datetime object into a string."""
    return dt.strftime("%Y-%m-%d %H:%M:%S %Z")

get_current_datetime("America/Los_Angeles")  # Example usage for PST

'2026-05-26 14:00:15 PDT'

In [78]:
get_current_datetime_schema = {
        "name": "get_current_datetime",
        "description": "Returns the current date and time in the specified IANA timezone. Use this whenever the user asks about the current time, today's date, or needs a timestamp. Defaults to UTC if no timezone is provided.",
        "input_schema": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "IANA timezone name (e.g., 'America/Los_Angeles', 'Europe/London', 'Asia/Tokyo'). Defaults to 'UTC'."
                }
            },
            "required": []
        }
    }


In [79]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None, model="claude-haiku-4-5"):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [80]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [ ]:
def run_conversation(messages, model="claude-haiku-4-5", system=None, temperature=1.0, stop_sequences=[], tools=None):
    while True:
        response = chat(messages, tools=tools, model=model, system=system, temperature=temperature, stop_sequences=stop_sequences)

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return 

In [98]:
messages = []

add_user_message(
    messages,
    "What is the current time in PST format?",
)

system = (
        "Only responsed with what is returned by tool call. No mark donwn only raw text Example"
        "For example: "
        "   What is the current time in UTC? Also, what is the current time in PST format? : 2026-05-26 21:01:00 UTC and 2026-05-26 14:01:00 PDT"
        )

run_conversation(messages, system=RETENTION_EMAIL_SYSTEM_PROMPT, temperature=0.0)



2026-05-26 14:06:51 PDT
